In [0]:
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import col

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gr5069.hc3613.f1_predictions_model1 (
  driverId INT, 
  stop INT, 
  lap INT, 
  actual_ms DOUBLE, 
  predicted_ms DOUBLE
);

CREATE TABLE IF NOT EXISTS gr5069.hc3613.f1_predictions_model2 (
  driverId INT, 
  stop INT, 
  lap INT, 
  actual_ms DOUBLE, 
  predicted_ms DOUBLE
);

In [0]:
from sklearn.model_selection import train_test_split

# 加载数据并准备训练/测试集
pit_stops_df = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True).select("driverId", "stop", "lap", "milliseconds").dropna().toPandas()
X = pit_stops_df[["driverId", "stop", "lap"]]
y = pit_stops_df["milliseconds"].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import tempfile

# define a function to train and log the model
def train_f1_model(model_obj, run_name, params, table_name):
    with mlflow.start_run(run_name=run_name) as run:
        # 1. 
        if params: model_obj.set_params(**params)
        model_obj.fit(X_train, y_train) 
        preds = model_obj.predict(X_test)
        
        # 2. record the model
        mlflow.sklearn.log_model(model_obj, "f1-model")
        if params: mlflow.log_params(params)
        
        # 3. 4 metrics
        mse = mean_squared_error(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        rmse = np.sqrt(mse)
        mlflow.log_metrics({"mse": mse, "mae": mae, "r2": r2, "rmse": rmse})
        
        # 4. 2 artifacts
    
        summary_pd = pd.Series(preds).describe().to_frame(name="stats")
        with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as f:
            summary_pd.to_csv(f.name)
            mlflow.log_artifact(f.name, "prediction_summary.csv")
            
        # 5. 2 tables
        residuals = y_test - preds
        res_pd = pd.DataFrame({"residual": residuals})
        with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as f:
            res_pd.to_csv(f.name, index=False)
            mlflow.log_artifact(f.name, "residuals_data.csv")

        
        results_pd = X_test.copy()
        results_pd["actual_ms"] = y_test
        results_pd["predicted_ms"] = preds
        return results_pd

# run 1: RandomForest
rf_results = train_f1_model(RandomForestRegressor(), "RF_Final_Model", {"n_estimators": 100}, "gr5069.hc3613.f1_predictions_model1")

# run 2: LinearRegression
lr_results = train_f1_model(LinearRegression(), "LR_Final_Model", {}, "gr5069.hc3613.f1_predictions_model2")

In [0]:
# predict 1 
spark.createDataFrame(rf_results).write.format("delta").mode("overwrite").saveAsTable("gr5069.hc3613.f1_predictions_model1")

# predict 2
spark.createDataFrame(lr_results).write.format("delta").mode("overwrite").saveAsTable("gr5069.hc3613.f1_predictions_model2")

In [0]:
%sql
SELECT * FROM gr5069.hc3613.f1_predictions_model1 LIMIT 10;